In [1]:
# IN06: Failure Resilience

In [2]:
import os, json, time, random
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(model='gpt-4-turbo', api_key=OPENAI_API_KEY, temperature=0)
print('LLM ready:', llm.model_name)

LLM ready: gpt-4-turbo


## Failure Scenarios and Resilience Patterns

Agentic systems fail in ways that traditional software does not.
Three failure modes every production agent must handle:

| Failure Mode | What happens | Detection | Mitigation |
|---|---|---|---|
| **Timeout** | External tool or LLM call exceeds latency budget | `time.time()` comparison or `signal.alarm` | Retry with back-off; fallback to cached result |
| **Hallucination** | LLM returns plausible but incorrect tool arguments or facts | Check tool output against known schema; LLM-as-judge | Re-prompt; return 'I don't know'; escalate to human |
| **Lost context** | Message history exceeds context window; older messages truncated | Token count check before invoke | Summarise history; use external memory store |

### Failure Mode 1: Timeout with Exponential Back-off Retry

In [3]:
# Agent → Inventory API → wait maximum 2 sec → timeout → wait → retry → still fails → fallback

import threading

# In real production applications, timeout handling is often provided directly by:
#     HTTP clients,
#     async frameworks,
#     API gateways,
#     SDKs,
#     orchestration frameworks.
# For example, an HTTP client may directly support:
#     timeout=2
# So you may not manually create a Python thread for every API call.

# Timeout → retry
# Authentication failure → don't retry blindly
# Invalid SKU → fix input
# Server crashed → maybe retry

# Create a custom timeout exception
class ToolTimeoutError(Exception):
    pass

def call_with_timeout(fn, args: dict, timeout_sec: float = 2.0):
    result = [None]
    error  = [None]

    def target():
        try:
            result[0] = fn(**args)
        except Exception as e:
            error[0] = e

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        raise ToolTimeoutError(f'Tool call timed out after {timeout_sec}s')
    if error[0]:
        raise error[0]
    return result[0]

def retry_with_backoff(fn, args: dict, max_retries: int = 3, base_delay: float = 0.5):
    last_error = None
    for attempt in range(max_retries):
        try:
            return call_with_timeout(fn, args, timeout_sec=2.0)
        except ToolTimeoutError as e:
            last_error = e
            delay = base_delay * (2 ** attempt) # delay = 0.5 * 2^0 = 0.5, delay = 0.5 * 2^1 = 1.0, delay = 0.5 * 2^2 = 2.0
            print(f'  Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s...')
            time.sleep(delay)
        except Exception as e:
            raise
    raise ToolTimeoutError(f'All {max_retries} attempts failed. Last error: {last_error}')

# Simulate a slow Walmart inventory API
def slow_inventory_api(sku: str) -> str:
    time.sleep(3.0)  # Exceeds the 2s timeout
    return f'SKU {sku}: In stock'

def fast_inventory_api(sku: str) -> str:
    time.sleep(0.1)  # Well within timeout
    return f'SKU {sku}: In stock (24 units)'

print('Test 1: Slow API (should timeout and retry)...')
try:
    retry_with_backoff(slow_inventory_api, {'sku': 'GV-MILK-1G'}, max_retries=2, base_delay=0.2)
except ToolTimeoutError as e:
    print(f'  All retries exhausted: {e}')
    print('  Fallback: serving cached inventory data')

print()
print('Test 2: Fast API (should succeed)...')
result = retry_with_backoff(fast_inventory_api, {'sku': 'GV-MILK-1G'})
print(f'  Result: {result}')

Test 1: Slow API (should timeout and retry)...
  Attempt 1 failed: Tool call timed out after 2.0s. Retrying in 0.2s...
  Attempt 2 failed: Tool call timed out after 2.0s. Retrying in 0.4s...
  All retries exhausted: All 2 attempts failed. Last error: Tool call timed out after 2.0s
  Fallback: serving cached inventory data

Test 2: Fast API (should succeed)...
  Result: SKU GV-MILK-1G: In stock (24 units)


### Failure Mode 2: Hallucination Detection

In [4]:
VALID_SKUS = {'GV-MILK-1G', 'GV-BREAD-20', 'GV-EGGS-12', 'GV-BUTT-1', 'GV-CHKN-3'}

def detect_hallucinated_sku(llm_extracted_sku: str) -> dict:
    sku = llm_extracted_sku.strip().upper()
    is_valid = sku in VALID_SKUS
    return {
        'sku': sku,
        'valid': is_valid,
        'action': 'proceed' if is_valid else 'reprompt',
        'message': f'SKU {sku} is valid.' if is_valid else f'SKU {sku} not in catalog. Valid SKUs: {sorted(VALID_SKUS)}',
    }

def sku_grounded_response(query: str) -> str:
    # Step 1: LLM extracts a SKU
    extract_prompt = ('From this customer query, extract the product SKU if mentioned, '
                      'or infer the most likely SKU from: GV-MILK-1G, GV-BREAD-20, GV-EGGS-12, GV-BUTT-1, GV-CHKN-3. '
                      'Return only the SKU string, nothing else.')
    extracted = llm.invoke([SystemMessage(content=extract_prompt), HumanMessage(content=query)])
    sku_candidate = extracted.content.strip()

    # Step 2: Validate -- catch hallucinated SKUs before they hit the inventory API
    check = detect_hallucinated_sku(sku_candidate)
    print(f'  Extracted SKU : {sku_candidate}')
    print(f'  Validation    : {check["message"]}')
    print(f'  Action        : {check["action"]}')
    if check['action'] == 'reprompt':
        return 'I could not find that product in our catalog. Could you clarify which item you mean?'

    # Step 3: Only call the inventory API with a validated SKU
    inventory = {
        'GV-MILK-1G': 'In stock: 24 units', 'GV-CHKN-3': 'Out of stock',
    }
    return inventory.get(check['sku'], 'Inventory data not available.')

print('Test 1: Valid product query')
print('  Result:', sku_grounded_response('Is milk available?'))
print()
print('Test 2: Ambiguous query (LLM may hallucinate SKU)')
print('  Result:', sku_grounded_response('Do you have the XYZ-9999 item?'))

Test 1: Valid product query
  Extracted SKU : GV-MILK-1G
  Validation    : SKU GV-MILK-1G is valid.
  Action        : proceed
  Result: In stock: 24 units

Test 2: Ambiguous query (LLM may hallucinate SKU)
  Extracted SKU : GV-MILK-1G
  Validation    : SKU GV-MILK-1G is valid.
  Action        : proceed
  Result: In stock: 24 units


In [5]:
# To be continued...

### Failure Mode 3: Circuit Breaker Pattern

A circuit breaker stops calling a failing service after N consecutive failures.
It moves through three states: CLOSED (normal) → OPEN (blocking) → HALF-OPEN (testing).

This prevents cascading failures when a Walmart microservice is degraded.